# Project 1: Experiments (Task 5, and Task 6 if done)

This notebook is a skeleton. Keep its sections and fill them in.

- All agents are defined in `agents.py`. Import them from there; do not define agents in this notebook.
- Run every season with `play(agent, bay)` from `agents.py`.
- Document each step: what you run, why, and what you observe.
- The notebook must run from top to bottom without errors (Kernel, Restart and Run All).

In [2]:
import sys
import importlib
# Create a mock 'imp' module to support legacy packages and extensions in Python 3.13+
try:
    import imp
except ImportError:
    from types import ModuleType
    imp = ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fishing import FishingBay, ContextualBay, load_log, choices
from agents import play, RandomAgent, RandomContextualAgent  # add your own agents here

ModuleNotFoundError: No module named 'fishing'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## 0. Your team logs

Put the JSON log of each team member next to this notebook and list the files below.

In [ ]:
TEAM_LOG_FILES = [
    "json/fishing_log_soleiman.json",
    "json/fishing_log_syncakov.json",
    "json/fishing_log_sytcheva.json",
]
logs = [load_log(path) for path in TEAM_LOG_FILES]

# Example: the spots chosen and the catches of the first member in Season 1
# spots, catches = choices(logs[0], season=1)

## Example: one season played by the random agent

`FishingBay(season, seed)` creates a random bay. The same seed always gives the same bay,
so different agents can be compared on exactly the same bays.

In [ ]:
bay = FishingBay(season=1, seed=0)
agent = RandomAgent(seed=0)
total = play(agent, bay)
print(f"Total catch: {total} kg over {bay.days} days")
print("Spots chosen:", [spot for spot, catch in bay.history])

## 1. Random bays

Evaluate every agent on at least 1000 random bays for each season (1, 2 and 3), and report the distribution of the results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agents import play, RandomAgent, HumanRuleAgent
from fishing import FishingBay

# Define the number of random bays to test
NUM_BAYS = 1000

# We will store the results of our experiments here
eval_results = []


def calculate_dq(bay):
    info = bay.reveal()
    expected = 0

    for day, (spot, catch) in enumerate(bay.history, start=1):
        means = info["means"]

        if "change_day" in info and day >= info["change_day"]:
            means = info["means_after_change"]

        expected += means[spot]

    best = bay.best_possible()
    return 100 * expected / best

for season in [1, 2, 3]:
    # Evaluate the RandomAgent as a baseline
    random_agent_catches = []
    dq = []
    for seed in range(NUM_BAYS):
        bay = FishingBay(season=season, seed=seed)
        agent = RandomAgent(seed=seed)
        total_catch = play(agent, bay)
        random_agent_catches.append(total_catch)
        dq.append(calculate_dq(bay))

    # Save metrics
    eval_results.append({
        'Season': f'Season {season}',
        'Agent': 'RandomAgent',
        # 'Mean Catch': np.mean(random_agent_catches),
        # 'Std Dev': np.std(random_agent_catches),
        # 'Min': np.min(random_agent_catches),
        # 'Max': np.max(random_agent_catches)
        "Catch" : bay.total,
        "Expected Catch": round(bay.best_possible()*np.mean(dq)/100, 2),
        "DQ": round(np.mean(dq), 2)
    })

for season in [1, 2, 3]:
    # Evaluate the RandomAgent as a baseline
    human_rule_agent_catches = []
    dq = []
    for seed in range(NUM_BAYS):
        bay = FishingBay(season=season, seed=seed)
        agent = HumanRuleAgent(seed=seed)
        total_catch = play(agent, bay)
        random_agent_catches.append(total_catch)
        dq.append(calculate_dq(bay))

    # Save metrics
    eval_results.append({
        'Season': f'Season {season}',
        'Agent': 'HumanRuleAgent',
        # 'Mean Catch': np.mean(random_agent_catches),
        # 'Std Dev': np.std(random_agent_catches),
        # 'Min': np.min(random_agent_catches),
        # 'Max': np.max(random_agent_catches)
        "Catch" : bay.total,
        "Expected Catch": round(bay.best_possible()*np.mean(dq)/100, 2),
        "DQ": round(np.mean(dq), 2)
    })

df_eval = pd.DataFrame(eval_results)

display(df_eval)

## 2. Your team's bays

Run your agents on the bays played by your team members and compare with their results.
`FishingBay.from_log(log, season)` rebuilds the exact bay a member played, with the same catch on every day.

In [ ]:
# Example: rebuild the Season 1 bay of the first member
# bay = FishingBay.from_log(logs[0], season=1)

# Your code here

# calculate decision quality for each season and each member

def calculate_dq(bay):
    info = bay.reveal()
    expected = 0

    for day, (spot, catch) in enumerate(bay.history, start=1):
        means = info["means"]

        if "change_day" in info and day >= info["change_day"]:
            means = info["means_after_change"]

        expected += means[spot]

    best = bay.best_possible()
    return 100 * expected / best


results = []

for name in TEAM_LOG_FILES:
    log = load_log(name)
    # Extract 'xxx' from 'json/fishing_log_xxx.json'
    member_name = name.split("_")[-1].replace(".json", "")

    for season in [1, 2, 3]:
    # for season in [1]:
        bay = FishingBay.from_log(log, season)
        spots, catches = choices(log, season)

        for spot in spots:
            bay.fish(spot)

        dq = calculate_dq(bay)

        results.append({
            "Member": member_name,
            "Season": season,
            "Catch" : bay.total,
            "Expected Catch": bay.best_possible()*dq/100,
            "DQ": round(dq, 2)
        })

df = pd.DataFrame(results)

display(df)

## 3. Seasons 2 and 3

Identify the best agent for Season 2 and for Season 3, and explain why.
`FishingBay(season, seed, days=..., shift_day=...)` lets you experiment with longer seasons.

In [ ]:
# Your code here

## 4. One parameter

Study the effect of one parameter of one method.

In [ ]:
# Your code here

## 5. Part IV (optional): contextual bay

`ContextualBay(seed)` lasts 300 days. Each morning, your agent receives the conditions
`x = [wind_kmh, waves_m, tide_m]` before choosing a spot. Contextual agents are subclasses of
`ContextualAgent` in `agents.py`.

In [ ]:
bay = ContextualBay(seed=0)
total = play(RandomContextualAgent(seed=0), bay)
print(f"Random contextual agent: {total} kg over {bay.days} days")

# Your code here